We want dialogue to appear in the format;
character: dialogue
And we want to extract
Character
Spell

We want to translate the character IDs into character names. Character names are stoed in the characters file. no shit.


In [20]:
import pandas as pd
import re

# Load dialogue
dialogue = pd.read_csv(
    "dialogue.csv",
    sep=",",
    encoding="latin1",
    quoting=3,
    on_bad_lines="skip"
)

# Load characters
characters = pd.read_csv("Characters.csv", encoding="cp1252")
character_map = dict(zip(characters["Character ID"], characters["Character Name"]))

# Keep character name + dialogue together
dialogue["Character Name"] = dialogue["Character ID"].map(character_map)
dialogue["Dialogue Text"] = dialogue["Dialogue"].fillna("")

# Import spell notebook output
%run spells.ipynb

# Keep a clean spell lookup from the spells notebook
spells_df = df[["Spell Name", "Incantation", "Topic Label"]].copy()
spells_df = spells_df.dropna(subset=["Incantation"])
spells_df["Incantation"] = spells_df["Incantation"].astype(str).str.lower().str.strip()

def normalize_for_matching(text: str) -> str:
    # Normalize punctuation/whitespace so multi-word incantations match consistently.
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_incantation_pattern(incantation: str) -> re.Pattern:
    tokens = [re.escape(t) for t in incantation.split() if t]
    if not tokens:
        # Matches nothing for empty tokens; keeps logic simple downstream.
        return re.compile(r"a^", flags=re.IGNORECASE)
    # Allow one or more spaces between words after normalization.
    pattern = r"\b" + r"\s+".join(tokens) + r"\b"
    return re.compile(pattern, flags=re.IGNORECASE)

# Deduplicate by normalized incantation and build regex patterns once
spells_df = spells_df.drop_duplicates(subset=["Incantation"]).reset_index(drop=True)
spell_patterns = {
    row["Incantation"]: build_incantation_pattern(row["Incantation"])
    for _, row in spells_df.iterrows()
}

# Collect every spell use
usage_rows = []

for _, row in dialogue.iterrows():
    text = normalize_for_matching(row["Dialogue Text"])
    char_name = row["Character Name"]
    char_id = row["Character ID"]

    for incantation, pattern in spell_patterns.items():
        hits = len(pattern.findall(text))
        if hits > 0:
            spell_row = spells_df.loc[spells_df["Incantation"] == incantation].iloc[0]
            usage_rows.append({
                "Character ID": char_id,
                "Character Name": char_name,
                "Spell Name": spell_row["Spell Name"],
                "Incantation": incantation,
                "Topic Label": spell_row["Topic Label"],
                "Uses": hits
            })

usage_df = pd.DataFrame(usage_rows)

# Total uses per character + spell
character_spell_counts = (
    usage_df.groupby(
        ["Character Name", "Spell Name", "Incantation", "Topic Label"],
        as_index=False,
        dropna=False
    )["Uses"].sum()
)

# Unique spell count per character
unique_spell_counts = (
    usage_df.groupby("Character Name")["Spell Name"]
    .nunique()
    .reset_index(name="Unique Spells Used")
)

# Optional: total spell uses per character
total_spell_uses = (
    usage_df.groupby("Character Name")["Uses"]
    .sum()
    .reset_index(name="Total Spell Uses")
)

   Spell ID       Incantation                        Spell Name  \
0         1             Accio                   Summoning Charm   
1         2         Aguamenti                Water-Making Spell   
2         3  Alarte Ascendare  Launch an object up into the air   
3         4         Alohomora                   Unlocking Charm   
4         5     Arania Exumai            Spider repelling spell   

                  Effect     Light  
0      Summons an object       NaN  
1         Conjures water  Icy blue  
2  Rockets target upward       Red  
3         Unlocks target      Blue  
4         Repels spiders      Blue  
['Spell ID', 'Incantation', 'Spell Name', 'Effect', 'Light']
['Summoning Charm' 'Water-Making Spell' 'Launch an object up into the air'
 'Unlocking Charm' 'Spider repelling spell' 'Slowing Charm'
 'Killing Curse' 'Exploding Charm' 'Brackium Emendo' 'Cistem Aperio'
 'Locking Spell' 'Blasting Curse' 'Cruciatus Curse' 'Severing Charm'
 'Dissendium' 'Engorgement Charm' 'Episke

In [21]:
#print(character_spell_counts.head(3))
#print(unique_spell_counts.head(3))
#print(total_spell_uses.head(20))
# Show spell uses for character ID 1 (Harry Potter): total spells used and unique spells used.
target_id = 1
target_name = character_map.get(target_id, "Unknown")

harry_spells = character_spell_counts[
    character_spell_counts["Character Name"] == target_name
].copy()

total_spells_used = int(harry_spells["Uses"].sum())
unique_spells_used = int(harry_spells["Spell Name"].nunique())

print(f"{target_name} (ID {target_id})")
print(f"Total spells used: {total_spells_used}")
print(f"Unique spells used: {unique_spells_used}")
# Give me a sorted list of his most used spells
harry_spells_sorted = harry_spells.sort_values(by="Uses", ascending=False)
print(harry_spells_sorted[["Spell Name", "Incantation", "Uses"]].head(10))

Harry Potter (ID 1)
Total spells used: 58
Unique spells used: 24
                Spell Name       Incantation  Uses
39     Wand-Lighting Charm             lumos    11
26          Patronus Charm  expecto patronum    10
25            Lumos Maxima      lumos maxima     5
35         Summoning Charm             accio     4
18         Disarming Charm      expelliarmus     3
34          Stunning Spell           stupefy     3
33  Spider repelling spell     arania exumai     2
16          Blasting Curse         confringo     2
31            Shield Charm           protego     2
29            Sectumsempra      sectumsempra     2
